In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)


# =====================================================
# FILE SETTINGS
# =====================================================

INPUT_FILE = "Fab Critical Parts AI Training.csv"

OUTPUT_FOLDER = Path(
    "forecast_results"
)

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True,
)


# =====================================================
# FORECAST SETTINGS
# =====================================================

# Historical test examples:
#
# 2026-01-01 -> Jan-Jun
# 2026-02-01 -> Feb-Jul
# 2026-03-01 -> Mar-Aug
# 2026-04-01 -> Apr-Sep

FORECAST_CUTOFF = "2026-10-01"


FORECAST_HORIZONS = [
    1,
    2,
    3,
    4,
    5,
    6,
]


# =====================================================
# HISTORICAL ACCURACY
# =====================================================

ACCURACY_LOOKBACK_MONTHS = 12


# =====================================================
# COLUMNS
# =====================================================

TARGET_COLUMN = (
    "Monthly Inventory Issues"
)

PART_COLUMN = "fpartno"

DATE_COLUMN = "Date"

COMMITS_COLUMN = (
    "Avg Commits/Month"
)

USAGE_CATEGORY_COLUMN = (
    "Usage Notes"
)


# =====================================================
# STANDARDIZED USAGE CATEGORIES
# =====================================================

COMMON_HIGH = "COMMON HIGH"

COMMON_LOW = "COMMON LOW"

DECREASING_COMMON = (
    "DECREASING COMMON"
)

NEW_COMMON = (
    "NEW COMMON"
)


# =====================================================
# V8 SETTINGS
# =====================================================

LAGS = [
    1,
    2,
    3,
    6,
    12,
]

MIN_NONZERO_MONTHS = 18


# =====================================================
# VERSION 13 TRANSITION SETTINGS
# =====================================================

# Only recent history matters for
# phase-in / phase-out parts.

TRANSITION_LOOKBACK_MONTHS = 6

TRANSITION_BASE_MONTHS = 3


# =====================================================
# NEW COMMON SETTINGS
# =====================================================

# Continue moderating observed growth.

TRANSITION_TREND_DAMPING = 0.60


# =====================================================
# DECREASING COMMON SETTINGS
# =====================================================

# A known phase-out part must decline
# by AT LEAST this amount per month,
# even when the latest 6 months happen
# to look flat or temporarily increase.

MIN_DECREASING_MONTHLY_DECAY = 0.10


# Prevent an unusually sharp recent
# drop from causing the forecast to
# collapse unrealistically fast.

MAX_DECREASING_MONTHLY_DECAY = 0.35


# Moderate the decline observed in
# recent history before applying the
# min/max limits.

DECREASING_DECAY_DAMPING = 0.60


print(
    "Version 13 settings loaded."
)

print(
    f"Forecast cutoff: "
    f"{FORECAST_CUTOFF}"
)

print(
    "\nRouting:"
)

print(
    "Common High / Common Low "
    "-> V8 recursive"
)

print(
    "Decreasing Common "
    "-> recent level + "
    "monthly phase-out decay"
)

print(
    "New Common "
    "-> recent growth"
)

print(
    "\nDecreasing Common decay:"
)

print(
    f"Minimum monthly decay: "
    f"{MIN_DECREASING_MONTHLY_DECAY:.0%}"
)

print(
    f"Maximum monthly decay: "
    f"{MAX_DECREASING_MONTHLY_DECAY:.0%}"
)

In [ ]:
data = pd.read_csv(
    INPUT_FILE
)


required_columns = {
    PART_COLUMN,
    DATE_COLUMN,
    TARGET_COLUMN,
    COMMITS_COLUMN,
    USAGE_CATEGORY_COLUMN,
}


missing_columns = (
    required_columns
    .difference(
        data.columns
    )
)


if missing_columns:

    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )


# =====================================================
# DATE
# =====================================================

data[DATE_COLUMN] = (
    pd.to_datetime(
        data[DATE_COLUMN],
        errors="raise",
    )
)


data = data[
    data[DATE_COLUMN]
    >= pd.Timestamp(
        "2023-01-01"
    )
].copy()


# =====================================================
# USAGE
# =====================================================

data[TARGET_COLUMN] = (
    pd.to_numeric(
        data[TARGET_COLUMN],
        errors="raise",
    )
)


# =====================================================
# COMMITMENTS
# =====================================================

data[COMMITS_COLUMN] = (
    pd.to_numeric(
        data[COMMITS_COLUMN],
        errors="coerce",
    )
    .fillna(0)
)


# =====================================================
# PART NUMBER
# =====================================================

data[PART_COLUMN] = (
    data[PART_COLUMN]
    .astype(str)
    .str.strip()
)


# =====================================================
# USAGE NOTES
#
# Raw values may look like:
#
# Common- High
# Common-Low (<10 per month)
# Decreasing Common
# New Common
# =====================================================

data[USAGE_CATEGORY_COLUMN] = (
    data[USAGE_CATEGORY_COLUMN]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)


def normalize_usage_category(
    value
):

    if (
        "COMMON" in value
        and "HIGH" in value
    ):
        return COMMON_HIGH

    if (
        "COMMON" in value
        and "LOW" in value
    ):
        return COMMON_LOW

    if (
        "DECREASING" in value
        and "COMMON" in value
    ):
        return DECREASING_COMMON

    if (
        "NEW" in value
        and "COMMON" in value
    ):
        return NEW_COMMON

    return "UNKNOWN"


data["Usage Category"] = (
    data[
        USAGE_CATEGORY_COLUMN
    ]
    .apply(
        normalize_usage_category
    )
)


# =====================================================
# CLEAN / SORT
# =====================================================

data = (
    data
    .dropna(
        subset=[
            PART_COLUMN,
            DATE_COLUMN,
            TARGET_COLUMN,
        ]
    )
    .sort_values(
        [
            PART_COLUMN,
            DATE_COLUMN,
        ]
    )
    .reset_index(
        drop=True
    )
)


# =====================================================
# DUPLICATE CHECK
# =====================================================

duplicates = data.duplicated(
    [
        PART_COLUMN,
        DATE_COLUMN,
    ],
    keep=False,
)


if duplicates.any():

    duplicate_rows = data.loc[
        duplicates,
        [
            PART_COLUMN,
            DATE_COLUMN,
        ],
    ]

    raise ValueError(
        "Duplicate part/month rows "
        "were found:\n"
        f"{duplicate_rows.head(20)}"
    )


print(
    f"Rows loaded: "
    f"{len(data):,}"
)

print(
    f"Unique parts: "
    f"{data[PART_COLUMN].nunique():,}"
)

print(
    f"First date: "
    f"{data[DATE_COLUMN].min()}"
)

print(
    f"Last source date: "
    f"{data[DATE_COLUMN].max()}"
)


print(
    "\nNormalized Usage Categories:"
)

print(
    data[
        "Usage Category"
    ]
    .value_counts(
        dropna=False
    )
)

In [ ]:
forecast_cutoff = pd.Timestamp(
    FORECAST_CUTOFF
)


# =====================================================
# PRESERVE ORIGINAL USAGE
# =====================================================

data["Original Usage"] = (
    data[TARGET_COLUMN]
    .copy()
)

data["Usage Outlier"] = False


# =====================================================
# ONLY HISTORY KNOWN BEFORE CUTOFF
# =====================================================

historical_mask = (
    data[DATE_COLUMN]
    < forecast_cutoff
)


historical_data_for_outliers = (
    data.loc[
        historical_mask
    ]
    .copy()
)


historical_data_for_outliers[
    "_usage_year"
] = (
    historical_data_for_outliers[
        DATE_COLUMN
    ]
    .dt.year
)


# =====================================================
# IDENTIFY EXTREME OUTLIERS
# =====================================================

def identify_extreme_outliers(
    part_data
):

    part_data = (
        part_data.copy()
    )

    usage = (
        part_data[
            TARGET_COLUMN
        ]
        .astype(float)
    )

    nonzero_usage = (
        usage[
            usage > 0
        ]
    )


    if len(nonzero_usage) < 6:
        return part_data


    q1 = nonzero_usage.quantile(
        0.25
    )

    q3 = nonzero_usage.quantile(
        0.75
    )

    iqr = q3 - q1


    iqr_upper_limit = (
        q3
        +
        (3 * iqr)
    )


    normal_median = (
        nonzero_usage.median()
    )


    median_upper_limit = (
        normal_median
        * 10
    )


    outlier_mask = (
        (usage > iqr_upper_limit)
        &
        (usage > median_upper_limit)
    )


    part_data.loc[
        outlier_mask,
        "Usage Outlier"
    ] = True


    return part_data


historical_data_for_outliers = (
    historical_data_for_outliers
    .groupby(
        PART_COLUMN,
        group_keys=False,
    )
    .apply(
        identify_extreme_outliers
    )
    .reset_index(
        drop=True
    )
)


# =====================================================
# NORMAL YEARLY AVERAGE
# =====================================================

normal_usage_rows = (
    historical_data_for_outliers[
        historical_data_for_outliers[
            "Usage Outlier"
        ] == False
    ]
    .copy()
)


yearly_normal_average = (
    normal_usage_rows
    .groupby(
        [
            PART_COLUMN,
            "_usage_year",
        ]
    )[TARGET_COLUMN]
    .mean()
    .reset_index(
        name=(
            "Yearly Normal Average Usage"
        )
    )
)


historical_data_for_outliers = (
    historical_data_for_outliers
    .merge(
        yearly_normal_average,
        on=[
            PART_COLUMN,
            "_usage_year",
        ],
        how="left",
    )
)


# =====================================================
# FALLBACK PART AVERAGE
# =====================================================

part_normal_average = (
    normal_usage_rows
    .groupby(
        PART_COLUMN
    )[TARGET_COLUMN]
    .mean()
    .reset_index(
        name=(
            "Part Normal Average Usage"
        )
    )
)


historical_data_for_outliers = (
    historical_data_for_outliers
    .merge(
        part_normal_average,
        on=PART_COLUMN,
        how="left",
    )
)


# =====================================================
# REPLACE OUTLIERS
# =====================================================

outlier_mask_history = (
    historical_data_for_outliers[
        "Usage Outlier"
    ] == True
)


historical_data_for_outliers.loc[
    outlier_mask_history,
    TARGET_COLUMN
] = (
    historical_data_for_outliers.loc[
        outlier_mask_history,
        "Yearly Normal Average Usage",
    ]
    .fillna(
        historical_data_for_outliers.loc[
            outlier_mask_history,
            "Part Normal Average Usage",
        ]
    )
)


# =====================================================
# WRITE CLEAN VALUES BACK
# =====================================================

cleaned_history = (
    historical_data_for_outliers[
        [
            PART_COLUMN,
            DATE_COLUMN,
            TARGET_COLUMN,
            "Usage Outlier",
        ]
    ]
    .copy()
)


data = (
    data
    .drop(
        columns=[
            "Usage Outlier"
        ],
        errors="ignore",
    )
    .merge(
        cleaned_history,
        on=[
            PART_COLUMN,
            DATE_COLUMN,
        ],
        how="left",
        suffixes=(
            "",
            "_cleaned",
        ),
    )
)


data[TARGET_COLUMN] = (
    data[
        f"{TARGET_COLUMN}_cleaned"
    ]
    .fillna(
        data[TARGET_COLUMN]
    )
)


data["Usage Outlier"] = (
    data[
        "Usage Outlier"
    ]
    .fillna(False)
    .astype(bool)
)


data = data.drop(
    columns=[
        f"{TARGET_COLUMN}_cleaned"
    ],
    errors="ignore",
)


# =====================================================
# OUTLIER REVIEW
# =====================================================

outlier_review = (
    data[
        data[
            "Usage Outlier"
        ] == True
    ][
        [
            PART_COLUMN,
            DATE_COLUMN,
            "Original Usage",
            TARGET_COLUMN,
        ]
    ]
    .copy()
    .rename(
        columns={
            TARGET_COLUMN:
                "Usage Used By Model"
        }
    )
)


print(
    f"Extreme historical outliers "
    f"corrected: "
    f"{len(outlier_review):,}"
)


if not outlier_review.empty:

    display(
        outlier_review
        .sort_values(
            [
                PART_COLUMN,
                DATE_COLUMN,
            ]
        )
        .reset_index(
            drop=True
        )
    )


# =====================================================
# CATEGORY-SPECIFIC HISTORY FILTER
# =====================================================
# Categories come from the normalized Usage Notes in Cell 1.
# Use the same latest supplied category as the forecast router.
# Only usage before the forecast cutoff counts toward eligibility.
# New/Decreasing Common use the existing six-row transition method;
# all other categories still require 18 nonzero usage months.

history_before_forecast = data.loc[
    data[DATE_COLUMN] < forecast_cutoff
].copy()

part_categories = (
    data.sort_values([PART_COLUMN, DATE_COLUMN])
    .groupby(PART_COLUMN)["Usage Category"]
    .last()
    .rename("Usage Category")
    .reset_index()
)

history_counts = (
    history_before_forecast.groupby(PART_COLUMN)[DATE_COLUMN]
    .nunique()
    .rename("Historical Months")
    .reset_index()
)

nonzero_history = (
    history_before_forecast.loc[
        history_before_forecast[TARGET_COLUMN] > 0
    ]
    .groupby(PART_COLUMN)[DATE_COLUMN]
    .nunique()
    .rename("Nonzero Usage Months")
    .reset_index()
)

part_history_check = (
    part_categories
    .merge(history_counts, on=PART_COLUMN, how="left")
    .merge(nonzero_history, on=PART_COLUMN, how="left")
)

for column in ["Historical Months", "Nonzero Usage Months"]:
    part_history_check[column] = (
        part_history_check[column].fillna(0).astype(int)
    )

transition_mask = part_history_check["Usage Category"].isin([
    NEW_COMMON, DECREASING_COMMON,
])

part_history_check["History Requirement"] = np.where(
    transition_mask,
    f"{TRANSITION_LOOKBACK_MONTHS} historical months (zero usage allowed)",
    f"{MIN_NONZERO_MONTHS} nonzero usage months",
)

part_history_check["Eligible"] = np.where(
    transition_mask,
    part_history_check["Historical Months"] >= TRANSITION_LOOKBACK_MONTHS,
    part_history_check["Nonzero Usage Months"] >= MIN_NONZERO_MONTHS,
)

eligible_parts = part_history_check.loc[
    part_history_check["Eligible"], PART_COLUMN
]
excluded_parts = part_history_check.loc[
    ~part_history_check["Eligible"]
].copy()

original_part_count = data[PART_COLUMN].nunique()

data = (
    data.loc[data[PART_COLUMN].isin(eligible_parts)]
    .copy()
    .sort_values([PART_COLUMN, DATE_COLUMN])
    .reset_index(drop=True)
)

print(f"\nOriginal parts: {original_part_count:,}")
print(f"Parts included: {data[PART_COLUMN].nunique():,}")
print(f"Parts excluded: {len(excluded_parts):,}")
print(
    "Transition parts included with fewer than "
    f"{MIN_NONZERO_MONTHS} nonzero months: "
    f"{(transition_mask & part_history_check['Eligible'] & (part_history_check['Nonzero Usage Months'] < MIN_NONZERO_MONTHS)).sum():,}"
)

if not excluded_parts.empty:
    display(excluded_parts)


In [ ]:
V8_FEATURE_COLUMNS = [

    "month",
    "year",
    "quarter",
    "time_idx",

    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_12",

    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_12",

    "rolling_median_3",
    "rolling_median_6",
    "rolling_median_12",

    "rolling_total_12",

    "rolling_std_3",
    "rolling_std_6",
    "rolling_std_12",

    "rolling_min_12",
    "rolling_max_12",

    "zero_month_percentage_12",
    "coefficient_variation_12",
    "recent_vs_annual",

    "trend_3",
    "trend_6",

    "commits_lag_1",
    "commits_lag_2",
    "commits_lag_3",
    "commits_lag_6",

    "commits_rolling_mean_3",
    "commits_rolling_mean_6",
    "commits_rolling_mean_12",

    "commits_rolling_sum_3",
    "commits_rolling_sum_6",

    "commits_recent_vs_annual",
]


def create_training_features(
    historical_data
):

    feature_data = (
        historical_data.copy()
    )


    feature_data["month"] = (
        feature_data[
            DATE_COLUMN
        ].dt.month
    )

    feature_data["year"] = (
        feature_data[
            DATE_COLUMN
        ].dt.year
    )

    feature_data["quarter"] = (
        feature_data[
            DATE_COLUMN
        ].dt.quarter
    )

    feature_data["time_idx"] = (
        np.arange(
            len(feature_data)
        )
    )


    for lag in LAGS:

        feature_data[
            f"lag_{lag}"
        ] = (
            feature_data[
                TARGET_COLUMN
            ]
            .shift(lag)
        )


    prior_usage = (
        feature_data[
            TARGET_COLUMN
        ]
        .shift(1)
    )


    feature_data[
        "rolling_mean_3"
    ] = prior_usage.rolling(3).mean()

    feature_data[
        "rolling_mean_6"
    ] = prior_usage.rolling(6).mean()

    feature_data[
        "rolling_mean_12"
    ] = prior_usage.rolling(12).mean()


    feature_data[
        "rolling_median_3"
    ] = prior_usage.rolling(3).median()

    feature_data[
        "rolling_median_6"
    ] = prior_usage.rolling(6).median()

    feature_data[
        "rolling_median_12"
    ] = prior_usage.rolling(12).median()


    feature_data[
        "rolling_total_12"
    ] = prior_usage.rolling(12).sum()


    feature_data[
        "rolling_std_3"
    ] = prior_usage.rolling(3).std()

    feature_data[
        "rolling_std_6"
    ] = prior_usage.rolling(6).std()

    feature_data[
        "rolling_std_12"
    ] = prior_usage.rolling(12).std()


    feature_data[
        "rolling_min_12"
    ] = prior_usage.rolling(12).min()

    feature_data[
        "rolling_max_12"
    ] = prior_usage.rolling(12).max()


    feature_data[
        "trend_3"
    ] = (
        feature_data["lag_1"]
        -
        feature_data["lag_3"]
    )


    feature_data[
        "trend_6"
    ] = (
        feature_data["lag_1"]
        -
        feature_data["lag_6"]
    )


    feature_data[
        "zero_month_percentage_12"
    ] = (
        prior_usage
        .rolling(12)
        .apply(
            lambda values:
                (values == 0).mean(),
            raw=True,
        )
    )


    feature_data[
        "coefficient_variation_12"
    ] = (
        feature_data[
            "rolling_std_12"
        ]
        /
        feature_data[
            "rolling_mean_12"
        ].replace(
            0,
            np.nan,
        )
    )


    feature_data[
        "recent_vs_annual"
    ] = (
        feature_data[
            "rolling_mean_3"
        ]
        -
        feature_data[
            "rolling_mean_12"
        ]
    )


    prior_commits = (
        feature_data[
            COMMITS_COLUMN
        ]
        .shift(1)
    )


    feature_data[
        "commits_lag_1"
    ] = (
        feature_data[
            COMMITS_COLUMN
        ].shift(1)
    )

    feature_data[
        "commits_lag_2"
    ] = (
        feature_data[
            COMMITS_COLUMN
        ].shift(2)
    )

    feature_data[
        "commits_lag_3"
    ] = (
        feature_data[
            COMMITS_COLUMN
        ].shift(3)
    )

    feature_data[
        "commits_lag_6"
    ] = (
        feature_data[
            COMMITS_COLUMN
        ].shift(6)
    )


    feature_data[
        "commits_rolling_mean_3"
    ] = (
        prior_commits
        .rolling(3)
        .mean()
    )

    feature_data[
        "commits_rolling_mean_6"
    ] = (
        prior_commits
        .rolling(6)
        .mean()
    )

    feature_data[
        "commits_rolling_mean_12"
    ] = (
        prior_commits
        .rolling(12)
        .mean()
    )


    feature_data[
        "commits_rolling_sum_3"
    ] = (
        prior_commits
        .rolling(3)
        .sum()
    )

    feature_data[
        "commits_rolling_sum_6"
    ] = (
        prior_commits
        .rolling(6)
        .sum()
    )


    feature_data[
        "commits_recent_vs_annual"
    ] = (
        feature_data[
            "commits_rolling_mean_3"
        ]
        -
        feature_data[
            "commits_rolling_mean_12"
        ]
    )


    training_rows = (
        feature_data
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna(
            subset=V8_FEATURE_COLUMNS
        )
        .reset_index(
            drop=True
        )
    )


    return (
        training_rows,
        V8_FEATURE_COLUMNS,
    )


print(
    "Exact V8 feature function created."
)

In [ ]:
def train_model(
    training_rows,
    feature_columns,
):

    sample_weights = np.linspace(
        1.0,
        2.0,
        len(training_rows),
    )


    model = LGBMRegressor(

        objective="poisson",

        n_estimators=250,

        learning_rate=0.05,

        max_depth=5,

        min_child_samples=10,

        reg_lambda=1.0,

        random_state=42,

        verbose=-1,
    )


    model.fit(

        training_rows[
            feature_columns
        ],

        training_rows[
            TARGET_COLUMN
        ],

        sample_weight=sample_weights,
    )


    return model


print(
    "Exact V8 model created."
)

In [ ]:
def forecast_future_months(
    model,
    historical_data,
    feature_columns,
    forecast_months,
    part_number,
):

    forecast_history = (
        historical_data[
            [
                DATE_COLUMN,
                TARGET_COLUMN,
                COMMITS_COLUMN,
            ]
        ]
        .copy()
        .sort_values(
            DATE_COLUMN
        )
        .reset_index(
            drop=True
        )
    )


    predictions = []


    for _ in range(
        forecast_months
    ):

        next_date = (
            forecast_history[
                DATE_COLUMN
            ].max()
            +
            pd.DateOffset(
                months=1
            )
        )


        recent_usage = (
            forecast_history[
                TARGET_COLUMN
            ]
        )

        recent_commits = (
            forecast_history[
                COMMITS_COLUMN
            ]
        )


        future_row = {

            "month":
                next_date.month,

            "year":
                next_date.year,

            "quarter":
                next_date.quarter,

            "time_idx":
                len(
                    forecast_history
                ),
        }


        for lag in LAGS:

            future_row[
                f"lag_{lag}"
            ] = (
                recent_usage
                .iloc[-lag]
            )


        last_3 = recent_usage.iloc[-3:]
        last_6 = recent_usage.iloc[-6:]
        last_12 = recent_usage.iloc[-12:]


        future_row[
            "rolling_mean_3"
        ] = last_3.mean()

        future_row[
            "rolling_mean_6"
        ] = last_6.mean()

        future_row[
            "rolling_mean_12"
        ] = last_12.mean()


        future_row[
            "rolling_median_3"
        ] = last_3.median()

        future_row[
            "rolling_median_6"
        ] = last_6.median()

        future_row[
            "rolling_median_12"
        ] = last_12.median()


        future_row[
            "rolling_total_12"
        ] = last_12.sum()


        future_row[
            "rolling_std_3"
        ] = last_3.std()

        future_row[
            "rolling_std_6"
        ] = last_6.std()

        future_row[
            "rolling_std_12"
        ] = last_12.std()


        future_row[
            "rolling_min_12"
        ] = last_12.min()

        future_row[
            "rolling_max_12"
        ] = last_12.max()


        future_row[
            "zero_month_percentage_12"
        ] = (
            last_12.eq(0).mean()
        )


        rolling_mean_12 = (
            future_row[
                "rolling_mean_12"
            ]
        )


        rolling_std_12 = (
            future_row[
                "rolling_std_12"
            ]
        )


        if rolling_mean_12 != 0:

            future_row[
                "coefficient_variation_12"
            ] = (
                rolling_std_12
                /
                rolling_mean_12
            )

        else:

            future_row[
                "coefficient_variation_12"
            ] = 0.0


        future_row[
            "recent_vs_annual"
        ] = (
            future_row[
                "rolling_mean_3"
            ]
            -
            future_row[
                "rolling_mean_12"
            ]
        )


        future_row[
            "trend_3"
        ] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-3]
        )


        future_row[
            "trend_6"
        ] = (
            recent_usage.iloc[-1]
            -
            recent_usage.iloc[-6]
        )


        last_commits_3 = (
            recent_commits.iloc[-3:]
        )

        last_commits_6 = (
            recent_commits.iloc[-6:]
        )

        last_commits_12 = (
            recent_commits.iloc[-12:]
        )


        future_row[
            "commits_lag_1"
        ] = recent_commits.iloc[-1]

        future_row[
            "commits_lag_2"
        ] = recent_commits.iloc[-2]

        future_row[
            "commits_lag_3"
        ] = recent_commits.iloc[-3]

        future_row[
            "commits_lag_6"
        ] = recent_commits.iloc[-6]


        future_row[
            "commits_rolling_mean_3"
        ] = last_commits_3.mean()

        future_row[
            "commits_rolling_mean_6"
        ] = last_commits_6.mean()

        future_row[
            "commits_rolling_mean_12"
        ] = last_commits_12.mean()

        future_row[
            "commits_rolling_sum_3"
        ] = last_commits_3.sum()

        future_row[
            "commits_rolling_sum_6"
        ] = last_commits_6.sum()


        future_row[
            "commits_recent_vs_annual"
        ] = (
            future_row[
                "commits_rolling_mean_3"
            ]
            -
            future_row[
                "commits_rolling_mean_12"
            ]
        )


        future_features = (
            pd.DataFrame(
                [future_row],
                columns=feature_columns,
            )
        )


        predicted_usage = float(
            model.predict(
                future_features
            )[0]
        )


        predicted_usage = max(
            0.0,
            predicted_usage,
        )


        predictions.append(
            {

                PART_COLUMN:
                    part_number,

                DATE_COLUMN:
                    next_date,

                "Predicted Usage":
                    predicted_usage,
            }
        )


        new_history_row = (
            pd.DataFrame(
                {

                    DATE_COLUMN:
                        [next_date],

                    TARGET_COLUMN:
                        [predicted_usage],

                    COMMITS_COLUMN:
                        [0],
                }
            )
        )


        forecast_history = (
            pd.concat(
                [
                    forecast_history,
                    new_history_row,
                ],
                ignore_index=True,
            )
        )


    return pd.DataFrame(
        predictions
    )


print(
    "Exact V8 recursive forecast "
    "function created."
)

In [ ]:
# =====================================================
# VERSION 13 TRANSITION FORECAST
#
# DECREASING COMMON
# -----------------
# Uses only recent actual history as the
# starting point.
#
# Forecast MUST decline because the
# business has identified the part as
# being phased down / phased out.
#
#
# NEW COMMON
# ----------
# Uses recent history and recent growth.
#
# Does not use recursive V8.
# =====================================================


def calculate_robust_recent_slope(
    usage_values
):

    values = np.asarray(
        usage_values,
        dtype=float,
    )


    slopes = []


    for start_index in range(
        len(values) - 1
    ):

        for end_index in range(
            start_index + 1,
            len(values),
        ):

            month_distance = (
                end_index
                -
                start_index
            )


            slope = (
                values[end_index]
                -
                values[start_index]
            ) / month_distance


            slopes.append(
                slope
            )


    if not slopes:

        return 0.0


    return float(
        np.median(
            slopes
        )
    )


def forecast_transition_part(
    historical_data,
    part_number,
    usage_category,
    forecast_cutoff,
    forecast_months=6,
):


    history = (
        historical_data
        .copy()
        .sort_values(
            DATE_COLUMN
        )
        .reset_index(
            drop=True
        )
    )


    if (
        len(history)
        <
        TRANSITION_LOOKBACK_MONTHS
    ):

        raise ValueError(
            "Not enough recent history "
            "for transition forecast."
        )


    # =====================================================
    # MOST RECENT 6 ACTUAL MONTHS
    # =====================================================

    recent_history = (
        history
        .tail(
            TRANSITION_LOOKBACK_MONTHS
        )
        .copy()
    )


    recent_usage = (
        recent_history[
            TARGET_COLUMN
        ]
        .astype(float)
        .reset_index(
            drop=True
        )
    )


    recent_3 = (
        recent_usage
        .tail(
            TRANSITION_BASE_MONTHS
        )
    )


    prior_3 = (
        recent_usage
        .iloc[
            :TRANSITION_BASE_MONTHS
        ]
    )


    # =====================================================
    # CURRENT DEMAND LEVEL
    #
    # Use the median of the latest 3 months
    # so one unusual month does not become
    # the entire starting point.
    # =====================================================

    current_level = float(
        recent_3.median()
    )


    # =====================================================
    # ROBUST RECENT TREND
    # =====================================================

    raw_recent_slope = (
        calculate_robust_recent_slope(
            recent_usage
        )
    )


    forecast_rows = []


    # =====================================================
    # DECREASING COMMON
    # =====================================================

    if (
        usage_category
        == DECREASING_COMMON
    ):


        # =================================================
        # COMPARE PRIOR 3 MONTHS TO RECENT 3 MONTHS
        #
        # Example:
        #
        # Prior 3 avg  = 100
        # Recent 3 avg = 60
        #
        # Recent demand is now 60% of the
        # prior recent demand.
        # =================================================

        prior_3_average = float(
            prior_3.mean()
        )


        recent_3_average = float(
            recent_3.mean()
        )


        # =================================================
        # CALCULATE OBSERVED 3-MONTH DECLINE
        # =================================================

        if (
            prior_3_average > 0
            and
            recent_3_average
            <
            prior_3_average
        ):

            three_month_retention = (
                recent_3_average
                /
                prior_3_average
            )


            # Convert the 3-month change
            # into an approximate monthly
            # retention rate.

            monthly_retention = (
                three_month_retention
                **
                (1 / 3)
            )


            observed_monthly_decay = (
                1.0
                -
                monthly_retention
            )


        else:

            # Recent data does not currently
            # show a decline.
            #
            # The business classification
            # still tells us the part is
            # being phased down.

            observed_monthly_decay = 0.0


        # =================================================
        # MODERATE OBSERVED DECAY
        # =================================================

        moderated_decay = (
            observed_monthly_decay
            *
            DECREASING_DECAY_DAMPING
        )


        # =================================================
        # GUARANTEE PHASE-OUT BEHAVIOR
        #
        # At least 10% decline/month.
        # At most 35% decline/month.
        # =================================================

        applied_decay = max(
            MIN_DECREASING_MONTHLY_DECAY,
            moderated_decay,
        )


        applied_decay = min(
            MAX_DECREASING_MONTHLY_DECAY,
            applied_decay,
        )


        method_name = (
            "Recent 6M Phase-Out Decay"
        )


        # =================================================
        # GENERATE COMPOUNDING DECLINE
        #
        # H1 = current level * (1 - decay)
        # H2 = H1 * (1 - decay)
        # etc.
        # =================================================

        for horizon in range(
            1,
            forecast_months + 1,
        ):


            target_date = (
                forecast_cutoff
                +
                pd.DateOffset(
                    months=(
                        horizon - 1
                    )
                )
            )


            predicted_usage = (
                current_level
                *
                (
                    (1.0 - applied_decay)
                    **
                    horizon
                )
            )


            # Very small phase-out values
            # can reasonably become zero.

            if predicted_usage < 0.5:

                predicted_usage = 0.0


            predicted_usage = max(
                0.0,
                predicted_usage,
            )


            forecast_rows.append(
                {

                    PART_COLUMN:
                        part_number,

                    "Forecast Origin":
                        history[
                            DATE_COLUMN
                        ].max(),

                    "Target Date":
                        target_date,

                    "Forecast Horizon":
                        horizon,

                    "Usage Category":
                        usage_category,

                    "Forecast Method":
                        method_name,

                    "Recent 6M Usage":
                        ", ".join(
                            str(
                                round(
                                    value,
                                    2,
                                )
                            )
                            for value
                            in recent_usage
                        ),

                    "Recent 3M Median":
                        current_level,

                    # Keep this column so the
                    # rest of V13 does not need
                    # to change.

                    "Raw Recent Trend":
                        raw_recent_slope,

                    # For Decreasing Common,
                    # show the approximate unit
                    # decrease that the applied
                    # decay causes in month 1.

                    "Applied Monthly Trend":
                        (
                            -current_level
                            *
                            applied_decay
                        ),

                    "Predicted Usage":
                        predicted_usage,
                }
            )


    # =====================================================
    # NEW COMMON
    #
    # Leave the existing growth logic unchanged
    # for this experiment.
    # =====================================================

    elif (
        usage_category
        == NEW_COMMON
    ):


        directional_slope = max(
            0.0,
            raw_recent_slope,
        )


        moderated_slope = (
            directional_slope
            *
            TRANSITION_TREND_DAMPING
        )


        method_name = (
            "Recent 6M Growth"
        )


        for horizon in range(
            1,
            forecast_months + 1,
        ):


            target_date = (
                forecast_cutoff
                +
                pd.DateOffset(
                    months=(
                        horizon - 1
                    )
                )
            )


            predicted_usage = (
                current_level
                +
                (
                    moderated_slope
                    *
                    horizon
                )
            )


            predicted_usage = max(
                0.0,
                predicted_usage,
            )


            forecast_rows.append(
                {

                    PART_COLUMN:
                        part_number,

                    "Forecast Origin":
                        history[
                            DATE_COLUMN
                        ].max(),

                    "Target Date":
                        target_date,

                    "Forecast Horizon":
                        horizon,

                    "Usage Category":
                        usage_category,

                    "Forecast Method":
                        method_name,

                    "Recent 6M Usage":
                        ", ".join(
                            str(
                                round(
                                    value,
                                    2,
                                )
                            )
                            for value
                            in recent_usage
                        ),

                    "Recent 3M Median":
                        current_level,

                    "Raw Recent Trend":
                        raw_recent_slope,

                    "Applied Monthly Trend":
                        moderated_slope,

                    "Predicted Usage":
                        predicted_usage,
                }
            )


    else:

        raise ValueError(
            "Unexpected transition "
            f"category: "
            f"{usage_category}"
        )


    return pd.DataFrame(
        forecast_rows
    )


print(
    "Version 13 transition "
    "forecast function created."
)

print(
    "Decreasing Common now uses "
    "compounding phase-out decay."
)

In [ ]:
historical_accuracy_rows = []

historical_accuracy_errors = []


forecast_cutoff = pd.Timestamp(
    FORECAST_CUTOFF
)


accuracy_end = (
    forecast_cutoff
    -
    pd.DateOffset(
        months=1
    )
)


accuracy_months = (
    pd.date_range(
        end=accuracy_end,
        periods=ACCURACY_LOOKBACK_MONTHS,
        freq="MS",
    )
)


part_numbers = sorted(
    data[
        PART_COLUMN
    ]
    .dropna()
    .unique()
)


total_parts = len(
    part_numbers
)


for part_index, part_number in enumerate(
    part_numbers,
    start=1,
):

    try:

        part_data = (
            data[
                data[
                    PART_COLUMN
                ] == part_number
            ]
            .copy()
            .sort_values(
                DATE_COLUMN
            )
            .reset_index(
                drop=True
            )
        )


        for target_month in (
            accuracy_months
        ):

            historical_data = (
                part_data[
                    part_data[
                        DATE_COLUMN
                    ]
                    < target_month
                ]
                .copy()
            )


            actual_row = (
                part_data[
                    part_data[
                        DATE_COLUMN
                    ]
                    == target_month
                ]
            )


            if actual_row.empty:
                continue


            (
                training_rows,
                feature_columns,
            ) = (
                create_training_features(
                    historical_data
                )
            )


            if training_rows.empty:
                continue


            if (
                training_rows[
                    TARGET_COLUMN
                ].sum()
                <= 0
            ):
                continue


            model = train_model(
                training_rows,
                feature_columns,
            )


            forecast = (
                forecast_future_months(

                    model=model,

                    historical_data=(
                        historical_data
                    ),

                    feature_columns=(
                        feature_columns
                    ),

                    forecast_months=1,

                    part_number=(
                        part_number
                    ),
                )
            )


            predicted_usage = float(
                forecast[
                    "Predicted Usage"
                ].iloc[0]
            )


            actual_usage = float(
                actual_row[
                    TARGET_COLUMN
                ].iloc[0]
            )


            error = (
                predicted_usage
                -
                actual_usage
            )


            historical_accuracy_rows.append(
                {

                    PART_COLUMN:
                        part_number,

                    "Target Month":
                        target_month,

                    "Predicted Usage":
                        predicted_usage,

                    "Actual Usage":
                        actual_usage,

                    "Error":
                        error,

                    "Absolute Error":
                        abs(
                            error
                        ),
                }
            )


    except Exception as error:

        historical_accuracy_errors.append(
            {

                PART_COLUMN:
                    part_number,

                "Error":
                    str(
                        error
                    ),
            }
        )


    if (
        part_index % 25 == 0
        or
        part_index == total_parts
    ):

        print(
            f"Historical accuracy: "
            f"{part_index:,} of "
            f"{total_parts:,} parts"
        )


historical_accuracy_detail = (
    pd.DataFrame(
        historical_accuracy_rows,
        columns=[
            PART_COLUMN,
            "Target Month",
            "Predicted Usage",
            "Actual Usage",
            "Error",
            "Absolute Error",
        ],
    )
)


if historical_accuracy_detail.empty:

    # Short-history transition parts may have no usable V8 backtests.
    # Keep forecasting; the left join will leave their accuracy blank.
    print(
        "No historical V8 accuracy rows were generated. "
        "Forecasting will continue with blank historical accuracy fields."
    )


print(
    "\nHistorical V8 accuracy "
    "calculation complete."
)

print(
    f"Forecast observations: "
    f"{len(historical_accuracy_detail):,}"
)

In [ ]:
historical_accuracy_summary = (
    historical_accuracy_detail
    .groupby(
        PART_COLUMN
    )
    .agg(

        Historical_Forecasts=(
            "Target Month",
            "count",
        ),

        Prior_12M_Predicted_Total=(
            "Predicted Usage",
            "sum",
        ),

        Prior_12M_Actual_Total=(
            "Actual Usage",
            "sum",
        ),

        Prior_12M_Total_Error=(
            "Error",
            "sum",
        ),

        Prior_12M_Absolute_Error=(
            "Absolute Error",
            "sum",
        ),

        Prior_12M_MAE=(
            "Absolute Error",
            "mean",
        ),

    )
    .reset_index()
)


historical_accuracy_summary[
    "Prior_12M_WAPE"
] = np.where(

    historical_accuracy_summary[
        "Prior_12M_Actual_Total"
    ] > 0,

    (
        historical_accuracy_summary[
            "Prior_12M_Absolute_Error"
        ]
        /
        historical_accuracy_summary[
            "Prior_12M_Actual_Total"
        ]
        * 100
    ),

    np.nan,
)


historical_accuracy_summary[
    "Prior_12M_Bias_Percent"
] = np.where(

    historical_accuracy_summary[
        "Prior_12M_Actual_Total"
    ] > 0,

    (
        historical_accuracy_summary[
            "Prior_12M_Total_Error"
        ]
        /
        historical_accuracy_summary[
            "Prior_12M_Actual_Total"
        ]
        * 100
    ),

    np.nan,
)


historical_accuracy_summary[
    "Historical_Direction"
] = np.where(

    historical_accuracy_summary[
        "Prior_12M_Total_Error"
    ] > 0,

    "Over Forecast",

    np.where(

        historical_accuracy_summary[
            "Prior_12M_Total_Error"
        ] < 0,

        "Under Forecast",

        "Neutral",
    ),
)


display(
    historical_accuracy_summary
    .sort_values(
        "Prior_12M_WAPE"
    )
)

In [ ]:
all_results = []

forecast_errors = []


forecast_cutoff = pd.Timestamp(
    FORECAST_CUTOFF
)


part_numbers = sorted(
    data[
        PART_COLUMN
    ]
    .dropna()
    .unique()
)


total_parts = len(
    part_numbers
)


for part_index, part_number in enumerate(
    part_numbers,
    start=1,
):

    try:

        part_data = (
            data[
                data[
                    PART_COLUMN
                ] == part_number
            ]
            .copy()
            .sort_values(
                DATE_COLUMN
            )
            .reset_index(
                drop=True
            )
        )


        historical_data = (
            part_data[
                part_data[
                    DATE_COLUMN
                ]
                < forecast_cutoff
            ]
            .copy()
            .sort_values(
                DATE_COLUMN
            )
            .reset_index(
                drop=True
            )
        )


        if historical_data.empty:

            raise ValueError(
                "No historical data "
                "before cutoff."
            )


        # =====================================================
        # USE STANDARDIZED CATEGORY
        # =====================================================

        category_values = (
            part_data[
                "Usage Category"
            ]
            .dropna()
        )


        if category_values.empty:

            usage_category = (
                "UNKNOWN"
            )

        else:

            usage_category = str(
                category_values.iloc[-1]
            ).strip().upper()


        # =====================================================
        # TRANSITION PARTS
        # =====================================================

        if usage_category in [
            DECREASING_COMMON,
            NEW_COMMON,
        ]:

            part_forecast = (
                forecast_transition_part(

                    historical_data=(
                        historical_data
                    ),

                    part_number=(
                        part_number
                    ),

                    usage_category=(
                        usage_category
                    ),

                    forecast_cutoff=(
                        forecast_cutoff
                    ),

                    forecast_months=6,
                )
            )


        # =====================================================
        # COMMON HIGH / COMMON LOW / OTHER
        # =====================================================

        else:

            (
                v8_training_rows,
                v8_feature_columns,
            ) = (
                create_training_features(
                    historical_data
                )
            )


            if v8_training_rows.empty:

                raise ValueError(
                    "No usable V8 "
                    "training rows."
                )


            if (
                v8_training_rows[
                    TARGET_COLUMN
                ].sum()
                <= 0
            ):

                raise ValueError(
                    "V8 training target "
                    "sum is zero."
                )


            v8_model = train_model(
                v8_training_rows,
                v8_feature_columns,
            )


            recursive_forecast = (
                forecast_future_months(

                    model=v8_model,

                    historical_data=(
                        historical_data
                    ),

                    feature_columns=(
                        v8_feature_columns
                    ),

                    forecast_months=6,

                    part_number=(
                        part_number
                    ),
                )
            )


            recursive_forecast = (
                recursive_forecast
                .reset_index(
                    drop=True
                )
            )


            recursive_forecast[
                "Forecast Horizon"
            ] = np.arange(
                1,
                len(
                    recursive_forecast
                ) + 1,
            )


            recursive_forecast[
                "Forecast Origin"
            ] = (
                historical_data[
                    DATE_COLUMN
                ].max()
            )


            recursive_forecast[
                "Target Date"
            ] = (
                recursive_forecast[
                    DATE_COLUMN
                ]
            )


            recursive_forecast[
                "Usage Category"
            ] = usage_category


            recursive_forecast[
                "Forecast Method"
            ] = np.where(

                recursive_forecast[
                    "Forecast Horizon"
                ] == 1,

                "Exact Version 8",

                "Recursive Version 8",
            )


            recursive_forecast[
                "Recent 6M Usage"
            ] = np.nan

            recursive_forecast[
                "Recent 3M Median"
            ] = np.nan

            recursive_forecast[
                "Raw Recent Trend"
            ] = np.nan

            recursive_forecast[
                "Applied Monthly Trend"
            ] = np.nan


            part_forecast = (
                recursive_forecast[
                    [
                        PART_COLUMN,

                        "Forecast Origin",
                        "Target Date",
                        "Forecast Horizon",

                        "Usage Category",
                        "Forecast Method",

                        "Recent 6M Usage",
                        "Recent 3M Median",
                        "Raw Recent Trend",
                        "Applied Monthly Trend",

                        "Predicted Usage",
                    ]
                ]
                .copy()
            )


        all_results.append(
            part_forecast
        )


    except Exception as error:

        forecast_errors.append(
            {

                PART_COLUMN:
                    part_number,

                "Error":
                    str(
                        error
                    ),
            }
        )


    if (
        part_index % 25 == 0
        or
        part_index == total_parts
    ):

        print(
            f"Forecasted "
            f"{part_index:,} of "
            f"{total_parts:,} parts"
        )


if not all_results:

    raise RuntimeError(
        "No Version 13 forecasts "
        "completed."
    )


print(
    "\nVersion 13 category-routed "
    "forecast complete."
)

print(
    f"Successful parts: "
    f"{len(all_results):,}"
)

print(
    f"Skipped parts: "
    f"{len(forecast_errors):,}"
)


if forecast_errors:

    display(
        pd.DataFrame(
            forecast_errors
        ).head(30)
    )

In [ ]:
results = (
    pd.concat(
        all_results,
        ignore_index=True,
    )
    .sort_values(
        [
            PART_COLUMN,
            "Target Date",
        ]
    )
    .reset_index(
        drop=True
    )
)


results[
    "Predicted Usage Rounded"
] = (
    results[
        "Predicted Usage"
    ]
    .round()
    .astype(
        "Int64"
    )
)


results = (
    results
    .merge(
        historical_accuracy_summary,
        on=PART_COLUMN,
        how="left",
    )
)


display(
    results[
        [
            PART_COLUMN,

            "Usage Category",

            "Forecast Origin",
            "Target Date",
            "Forecast Horizon",

            "Forecast Method",

            "Predicted Usage",
            "Predicted Usage Rounded",

            "Recent 6M Usage",
            "Recent 3M Median",
            "Raw Recent Trend",
            "Applied Monthly Trend",

            "Prior_12M_Predicted_Total",
            "Prior_12M_Actual_Total",
            "Prior_12M_WAPE",
            "Prior_12M_Bias_Percent",
            "Historical_Direction",
        ]
    ]
)

In [ ]:
actual_test_data = (
    data[
        (
            data[
                DATE_COLUMN
            ]
            >= forecast_cutoff
        )
        &
        (
            data[
                DATE_COLUMN
            ]
            <
            (
                forecast_cutoff
                +
                pd.DateOffset(
                    months=6
                )
            )
        )
    ][
        [
            PART_COLUMN,
            DATE_COLUMN,
            "Original Usage",
        ]
    ]
    .copy()
)


actual_test_data = (
    actual_test_data
    .rename(
        columns={

            DATE_COLUMN:
                "Target Date",

            "Original Usage":
                "Actual Usage",
        }
    )
)


evaluation = (
    results
    .merge(
        actual_test_data,
        on=[
            PART_COLUMN,
            "Target Date",
        ],
        how="left",
    )
)


evaluation[
    "Absolute Error"
] = (
    evaluation[
        "Predicted Usage"
    ]
    -
    evaluation[
        "Actual Usage"
    ]
).abs()


evaluation[
    "Percent Error"
] = np.where(

    evaluation[
        "Actual Usage"
    ] > 0,

    (
        evaluation[
            "Absolute Error"
        ]
        /
        evaluation[
            "Actual Usage"
        ]
        * 100
    ),

    np.nan,
)


display(
    evaluation[
        [
            PART_COLUMN,
            "Usage Category",

            "Target Date",
            "Forecast Horizon",

            "Forecast Method",

            "Actual Usage",
            "Predicted Usage Rounded",

            "Absolute Error",
            "Percent Error",
        ]
    ]
)

In [ ]:
category_performance = (
    evaluation
    .groupby(
        "Usage Category"
    )
    .agg(

        Forecasts=(
            PART_COLUMN,
            "count",
        ),

        Actual_Usage=(
            "Actual Usage",
            "sum",
        ),

        Predicted_Usage=(
            "Predicted Usage",
            "sum",
        ),

        Absolute_Error=(
            "Absolute Error",
            "sum",
        ),

        MAE=(
            "Absolute Error",
            "mean",
        ),

    )
    .reset_index()
)


category_performance[
    "WAPE"
] = np.where(

    category_performance[
        "Actual_Usage"
    ] > 0,

    (
        category_performance[
            "Absolute_Error"
        ]
        /
        category_performance[
            "Actual_Usage"
        ]
        * 100
    ),

    np.nan,
)


category_performance[
    "Total Bias"
] = (
    category_performance[
        "Predicted_Usage"
    ]
    -
    category_performance[
        "Actual_Usage"
    ]
)


category_performance[
    "Bias Percent"
] = np.where(

    category_performance[
        "Actual_Usage"
    ] > 0,

    (
        category_performance[
            "Total Bias"
        ]
        /
        category_performance[
            "Actual_Usage"
        ]
        * 100
    ),

    np.nan,
)


display(
    category_performance
    .sort_values(
        "WAPE"
    )
)

In [ ]:
category_horizon_performance = (
    evaluation
    .groupby(
        [
            "Usage Category",
            "Forecast Horizon",
        ]
    )
    .agg(

        Forecasts=(
            PART_COLUMN,
            "count",
        ),

        Actual_Usage=(
            "Actual Usage",
            "sum",
        ),

        Predicted_Usage=(
            "Predicted Usage",
            "sum",
        ),

        Absolute_Error=(
            "Absolute Error",
            "sum",
        ),

    )
    .reset_index()
)


category_horizon_performance[
    "WAPE"
] = np.where(

    category_horizon_performance[
        "Actual_Usage"
    ] > 0,

    (
        category_horizon_performance[
            "Absolute_Error"
        ]
        /
        category_horizon_performance[
            "Actual_Usage"
        ]
        * 100
    ),

    np.nan,
)


display(
    category_horizon_performance
)

In [ ]:
import boto3

from sagemaker_studio import Project

import io


proj = Project()


project_s3_root = (
    proj.s3.root
)


s3_parts = (
    project_s3_root
    .replace(
        "s3://",
        "",
    )
    .split(
        "/",
        1,
    )
)


bucket = (
    s3_parts[0]
)


prefix = (
    s3_parts[1]
    if len(s3_parts) > 1
    else ""
)


s3 = boto3.client(
    "s3"
)


results_to_upload = (
    results[
        [
            PART_COLUMN,

            "Usage Category",

            "Forecast Origin",
            "Target Date",
            "Forecast Horizon",

            "Forecast Method",

            "Predicted Usage",
            "Predicted Usage Rounded",

            "Recent 6M Usage",
            "Recent 3M Median",
            "Raw Recent Trend",
            "Applied Monthly Trend",

            "Prior_12M_Predicted_Total",
            "Prior_12M_Actual_Total",
            "Prior_12M_Total_Error",
            "Prior_12M_MAE",
            "Prior_12M_WAPE",
            "Prior_12M_Bias_Percent",
            "Historical_Direction",
        ]
    ]
    .copy()
)


csv_buffer = (
    io.StringIO()
)


results_to_upload.to_csv(
    csv_buffer,
    index=False,
)


cutoff_label = (
    pd.Timestamp(
        FORECAST_CUTOFF
    )
    .strftime(
        "%Y_%m"
    )
)


s3_key = (
    f"{prefix}/results/"
    f"version_13_{cutoff_label}_forecastfab3.csv"
)


s3.put_object(

    Bucket=bucket,

    Key=s3_key,

    Body=(
        csv_buffer
        .getvalue()
        .encode(
            "utf-8"
        )
    ),

    ContentType="text/csv",
)


s3_path = (
    f"s3://{bucket}/{s3_key}"
)


print(
    "Successfully uploaded "
    "Version 13 forecast!"
)

print(
    f"S3 path: "
    f"{s3_path}"
)

print(
    f"Rows uploaded: "
    f"{len(results_to_upload):,}"
)